# Example 7.10.2 / 1: A Reproducible Harness

Companion notebook for **7.10.2 Comparing Formulations Computationally** in the AMPL lab Documentation, section "A Reproducible Harness". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A Reproducible Harness

The activation model from Formulation Strength (AMPL lab documentation, section 7.10) contains an aggregate link and
disaggregated links; dropping one of them gives each formulation. The loop solves every
formulation on every instance with the same options and collects one row per run:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set I;                            # members of the group
param U {I} > 0;                  # upper bound of each member
param gain {I};                   # value per unit of each member
param fixed_cost >= 0;            # cost of switching the group on
var x {i in I} >= 0, <= U[i];     # activity of each member
var y binary;                     # 1 if the group is switched on

maximize Net: sum {i in I} gain[i] * x[i] - fixed_cost * y;
subject to TotalActivation: sum {i in I} x[i] <= (sum {i in I} U[i]) * y;  # aggregate
subject to Activation {i in I}: x[i] <= U[i] * y;                          # disaggregated

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["I"] = [1, 2, 3]
ampl.param["U"] = {1: 4, 2: 6, 3: 10}
instances = {"A": ({1: 5, 2: 0, 3: 0}, 15), "B": ({1: 3, 2: 2, 3: 1}, 25)}

In [5]:
# run.py
import re
import pandas as pd

formulations = {"aggregate": "Activation", "disaggregated": "TotalActivation"}
rows = []
for instance, (gain, fixed_cost) in instances.items():
    ampl.param["gain"] = gain
    ampl.param["fixed_cost"] = fixed_cost
    for name, other in formulations.items():
        ampl.con[other].drop()
        ampl.option["relax_integrality"] = 1
        ampl.solve(solver="highs", verbose=False)
        root = ampl.obj["Net"].value()
        ampl.option["relax_integrality"] = 0
        ampl.solve(solver="highs", highs_options="tech:seed=1", verbose=False)
        message = ampl.get_value("solve_message")
        rows.append({"instance": instance, "formulation": name,
                     "rows": ampl.get_value("_ncons"), "status": ampl.solve_result,
                     "root": root, "objective": ampl.obj["Net"].value(),
                     "nodes": int(re.search(r"(\d+) branching node", message).group(1))})
        ampl.con[other].restore()
print(pd.DataFrame(rows).to_string(index=False))

instance   formulation  rows status  root  objective  nodes
       A     aggregate     1 solved  17.0        5.0      0
       A disaggregated     3 solved   5.0        5.0      0
       B     aggregate     1 solved  11.5        9.0      0
       B disaggregated     3 solved   9.0        9.0      0
